# Kinetic Master Join: Building a Complete Space Object List

**Datasets:**
- SATCAT (Physical registry, ~67,000 objects)
- UCS (Active satellites, ~7,500 payloads)

**Objective:** Produce a single, easy-to-analyze dataset that brings together what we know about all tracked objects in orbit—both debris and active satellites.

### Why this notebook?
SATCAT tells us where things are, but not always what they do. UCS tells us about active satellites, but not debris. To understand risks in orbit, we need both worlds in one place. This notebook combines them.

### What we do here
1. **Join the data:** Match up active satellites from UCS with their physical details in SATCAT, keeping all debris too.
2. **Spot the "zombies":** Find satellites that are still in orbit but no longer working.
3. **Add physics:** Calculate speed and kinetic energy for each object.
4. **Save the result:** Output a single file for later analysis.

This step sets up everything needed for risk modeling and deeper questions later in the project.

In [1]:
import pandas as pd
import numpy as np
import utility as utils
from IPython.display import Markdown, display

### Stage 1: Merging SATCAT and UCS

**The issue:**
- SATCAT lists all tracked objects (including debris), but doesn’t say what each object does.
- UCS has details about active satellites, but not debris.
- If we just merge everything, we risk losing debris or duplicating columns.

**What we do:**
- Pick just what we need from UCS: Only keep the most useful columns (mission, status, etc.) to avoid clutter.
- Left join: Use SATCAT as the base, so we keep every object—including all debris and rocket bodies.
- Keep ALL objects including objeects that have deorbited.

**Why it matters:**
- This step gives us a single table with both physical and mission info, ready for the next stage.
- Ensures we don’t lose debris or important details, so our risk analysis is complete and reliable.

In [2]:
# We enforce string types for IDs immediately to prevent "Invisible Bug" merge failures
print("Loading Datasets...")
satcat = pd.read_csv('../data/clean/satcat_cleaned.csv', dtype={'norad_id': str, 'cospar_id': str}, low_memory=False)
ucs = pd.read_csv('../data/clean/ucs_cleaned.csv', dtype={'norad_id': str, 'cospar_id': str}, low_memory=False)

# Define the "Intelligence Packet"
# We strictly select only the metadata columns from UCS to avoid duplication.
# 'object_type' is NOT in this list because we rely on the SATCAT version.
ucs_intelligence_cols = [
    'norad_id',              # The Key
    'satellite_name',        # Human name
    'official_name',         # Full name
    'country_operator',      # Readable Country Name
    'users',                 # Sector string
    'primary_purpose',       # Standardized Mission (e.g., 'Communications')
    'detailed_purpose',      # Granular Mission
    'orbit_type',            # Geometry label (e.g., 'Polar')
    'is_commercial', 'is_government', 'is_military', 'is_civil', # Sector Flags
    'lifetime_years',        # Critical for Zombie Algorithm
    'sat_age_years',         # Satellite Age (years)
    'power_watts',           # Power (watts)
    'launch_mass_kg',        # Launch mass (kg)
    'dry_mass_kg',           # Dry mass (kg)
    'perigee_km',            # Perigee (km)
    'apogee_km',             # Apogee (km)
    'eccentricity',          # Orbit eccentricity
    'inclination_degrees',   # Inclination (deg)
    'period_minutes',        # Period (min)
    'un_registry',           # UN Registry Status
    'geo_longitude',         # GEO Slot (if applicable)
    'launch_site',           # Launch Site (if available)
    'contractor',            # Primary Contractor Name 
    'contractor_country'     # Contractor Country (if different from operator)
 ]

# Filter UCS down to just the intelligence packet
ucs_lean = ucs[ucs_intelligence_cols].copy()

# Execute Left Join
# SATCAT is the backbone (Left) so we keep all debris/rocket bodies
master = satcat.merge(ucs_lean, on='norad_id', how='left').copy()

# Normalize merge collisions for duplicate field names
# UCS values are prioritized over SATCAT for these fields.
for col in ucs_intelligence_cols:
    x_col, y_col = f'{col}_x', f'{col}_y'
    if y_col in master.columns or x_col in master.columns:
        master[col] = master.get(y_col).combine_first(master.get(x_col))
        master = master.drop(columns=[c for c in [x_col, y_col] if c in master.columns])

print(f"\n{'--- MERGE AUDIT ---':^40}")    
print(f"Kinetic Master (All):      {len(master):,}")
print("-" * 40)
print(f"Intelligence Matches:     {master['satellite_name'].notna().sum():,} (Active Payloads)")

# Check for leftover merge suffix columns
suffix_cols = [col for col in master.columns if col.endswith('_x') or col.endswith('_y')]
if suffix_cols:
    print("Warning: The following columns still have merge suffixes and may need manual review:", suffix_cols)
else:
    print("✅ No _x or _y columns remain after normalization.")

Loading Datasets...



          --- MERGE AUDIT ---           
Kinetic Master (All):      68,282
----------------------------------------
Intelligence Matches:     7,542 (Active Payloads)
✅ No _x or _y columns remain after normalization.


### Stage 2: Finding "Zombie" Satellites and Organizing Columns

**The issue:**
- Some satellites look active but are actually long dead—these "zombies" can confuse risk analysis.
- The raw object types don’t clearly separate active satellites from dead ones or debris.
- After merging, the columns are jumbled, making the table hard to read and use.

**What we do:**
- Flag satellites as "zombies" if they’re non-operational or have outlived their design life by 10% or more.
- Create a new "category" column to clearly label each object (Active Satellite, Inactive Satellite, Rocket Body, Debris).
- Reorder columns into logical groups so the table is easier to scan and analyze.

**Why it matters:**
- Makes it easy to spot dead satellites and avoid mistakes in risk modeling.
- Clean categories and organized columns help with charts, queries, and sharing results.
- Zombies are a major source of collision risk—if struck, they can create thousands of new debris fragments and accelerate the Kessler Syndrome.

In [3]:
# Initialize the Zombie Flag (Default to 0)
master['is_zombie'] = 0

# Define Zombie Logic
# This only applies to payloads (satellites). Debris and rocket bodies are excluded 
# from zombie classification by design. The payload filter ensures we only flag inactive/non-operational satellites
# instead of snagging all dead objects regardless of type.
# Expected result: ~5,200-5,300 zombie payloads

payload_mask = master['object_type'] == 'PAYLOAD'
status_zombie = payload_mask & master['ops_status'].isin(['NON-OPERATIONAL', 'PARTIAL', 'STANDBY', 'DECAYED', 'UNKNOWN'])

# this is a filter created from a compound conditional statement
# ( is_payload AND ops_status is OPERATIONAL AND sat_age > ( lifetime + 10% )
lifecycle_zombie = (
    payload_mask & 
    (master['ops_status'] == 'OPERATIONAL') & 
    (master['sat_age_years'] > (master['lifetime_years'] * 1.1))
)

# now we set is_zomebie = 1 for any rows that match the compound filter: [status_zombie | lifecycle_zombie]
# df.loc[COMPOUND | CONDITION, COLUMN] = VALUE
master.loc[status_zombie | lifecycle_zombie, 'is_zombie'] = 1

master['category'] = master.apply(utils.derive_category, axis=1)

master['owner'] = master['owner_code']

# all were doing here is reordering columns into their logical groups
# to make visual inspection of the dataframe easier
logical_order = [
    # --- IDENTITY ---
    'norad_id', 'cospar_id', 
    'object_name', 
    'satellite_name', 'official_name', 'category', 'object_type',
    
    # --- KINETIC PROFILE ---
    'launch_mass_kg', 
    'proxy_mass_kg', 
    'dry_mass_kg', 'power_watts', 'rcs', 'rcs_class',
    
    # --- ORBITAL STATE ---
    'velocity_kms', 'kinetic_joules', 'semi_major_axis_km', 'proxy_power_watts',
    'orbit_class', 'orbit_type', 'period_minutes', 
    'perigee_km', 'apogee_km', 'inclination_degrees', 'eccentricity',
    
    # --- MISSION INTELLIGENCE ---
    'primary_purpose', 'detailed_purpose', 'users', 'country_operator',
    'is_commercial', 'is_government', 'is_military', 'is_civil',
    'geo_longitude', 'un_registry',
    
    # --- LIFECYCLE & STATUS ---
    'launch_date', 'launch_year', 'sat_age_years', 'lifetime_years',
    'launch_site',
    'ops_status', 
    'data_status', 
    'decay_date', 'in_orbit', 'is_zombie',
    
    # --- SUPPLY CHAIN & METADATA ---
    'owner', 'owner_code', 'contractor', 'contractor_country'
]

# basically cross reference our logical_order column list with the actual columns in the master.
# as long as the column exists in the master, we keep it.

final_cols = [c for c in logical_order if c in master.columns]

# reassign master to itself but with the new column order. This doesn't drop any columns, just reorders them.
# I prefer C# and Javascript but I do like how flexible pandas is with this kind of thing. You can easily reorder, subset, 
# or even duplicate columns with simple list comprehensions.
master = master[final_cols]

# 5. Export
print(f"{'--- COMPATIBILITY AUDIT ---':^40}")
print(f"Object Name Density:      {master['object_name'].notna().mean():.1%} (Should be ~100%)")
print(f"Proxy Mass Present:       {'proxy_mass_kg' in master.columns}")
print(f"Final Schema Shape:       {master.shape[1]} columns")

output_path = '../data/clean/kinetic_master.csv'
master.to_csv(output_path, index=False)
print(f"\n✅ EXPORT SUCCESS: {len(master):,} active records saved to {output_path}")

      --- COMPATIBILITY AUDIT ---       
Object Name Density:      100.0% (Should be ~100%)
Proxy Mass Present:       True
Final Schema Shape:       44 columns



✅ EXPORT SUCCESS: 68,282 active records saved to ../data/clean/kinetic_master.csv


### Stage 3: Calculating Kinetic Energy and Power

**The issue:**
- Knowing where objects are isn’t enough—we need to know how much damage they could do if they collide.
- Many satellites are missing key physics details, making it hard to compare risks or fill in gaps for analysis.

**What we do:**
- Calculate each object’s orbital speed and kinetic energy using standard physics formulas.
- Estimate power capacity for satellites, using smart fill-in rules when data is missing.
- Add these new fields to the table for every object.

**Why it matters:**
- Kinetic energy tells us how destructive a collision could be—critical for risk modeling and Kessler Syndrome scenarios.
- Power estimates help us understand satellite capabilities and fill in missing data for better analysis.
- With these fields, we can compare objects on more than just location—they now have real-world impact and context.

In [4]:
# Define Astrodynamic Constants
MU = 398600.4418  # Standard Gravitational Parameter (km^3/s^2)

# Derive Semi-Major Axis (a) from Mean Motion
period_seconds = master['period_minutes'] * 60
mean_motion = (2 * np.pi) / period_seconds
master['semi_major_axis_km'] = np.cbrt(MU / mean_motion**2)

# Calculate Mean Orbital Velocity (km/s)
master['velocity_kms'] = np.sqrt(MU / master['semi_major_axis_km'])

# Calculate Kinetic Energy (Joules)
v_ms = master['velocity_kms'] * 1000
master['kinetic_joules'] = 0.5 * master['proxy_mass_kg'] * (v_ms ** 2)

# Initialize Proxy with Raw Data (Source of Truth)
master['proxy_power_watts'] = master['power_watts']

# Calculate "Tech Density" (Watts per Kg) for the KNOWN population
known_set = master.dropna(subset=['power_watts', 'proxy_mass_kg']).copy()
known_set['power_density'] = known_set['power_watts'] / known_set['proxy_mass_kg']

# Build the "Smart Lookup" Table (Median Density by Orbit Class)
# This captures the fact that GEO buses are fundamentally different from LEO cubesats.
orbit_density_map = known_set.groupby('orbit_class')['power_density'].median()
global_density = known_set['power_density'].median() # Fallback

print("--- SMART POWER MODEL PARAMETERS ---")
print(orbit_density_map)
print(f"Global Fallback: {global_density:.4f} W/kg")

# Apply the Smart Logic
master['proxy_power_watts'] = master.apply(utils.fill_power_smart, axis=1, args=(orbit_density_map, global_density))

# Final Schema Update
new_cols = ['velocity_kms', 'kinetic_joules', 'semi_major_axis_km', 'proxy_power_watts']
# Insert strictly into Kinetic Profile section
if 'rcs_class' in logical_order:
    insert_idx = logical_order.index('rcs_class') + 1
    final_schema = [c for c in logical_order if c not in new_cols]
    final_schema = final_schema[:insert_idx] + new_cols + final_schema[insert_idx:]
else:
    final_schema = master.columns.tolist()

master = master[[c for c in final_schema if c in master.columns]]

print(f"\n{'--- PHYSICS & ENGINEERING AUDIT ---':^50}")
print(f"Velocity Calculated:      {master['velocity_kms'].notna().mean():.1%}")
print(f"Kinetic Energy Calculated:{master['kinetic_joules'].notna().mean():.1%}")
print(f"Power Proxy Density:      {master['proxy_power_watts'].notna().mean():.1%} (Fully Filled for Known, Smart-Filled for Unknown)")

--- SMART POWER MODEL PARAMETERS ---
orbit_class
ELLIPTICAL    0.400000
GEO           2.181574
LEO           0.461538
MEO           1.081081
Name: power_density, dtype: float64
Global Fallback: 0.4615 W/kg



       --- PHYSICS & ENGINEERING AUDIT ---        
Velocity Calculated:      100.0%
Kinetic Energy Calculated:100.0%
Power Proxy Density:      100.0% (Fully Filled for Known, Smart-Filled for Unknown)


### Stage 4: Export and Final Report

**The issue:**
- After all processing, we need a clear, reliable way to check our results and share the finished dataset.
- Without a summary and validation, errors or missing data could go unnoticed.

**What we do:**
- Run a quick, standardized report to check data quality, completeness, and schema.
- Export the final, cleaned dataset to a CSV file for use in analysis, visualization, or sharing.
- Double-check the export by reading it back and confirming the record count.

**Why it matters:**
- The report gives confidence that the data is ready for analysis and meets project standards.
- Exporting to CSV makes the results portable and easy to use in other tools.
- This step ensures the project ends with a trustworthy, well-documented dataset.

In [5]:
utils.quick_report(master, title="Kinetic Master Post-Synthesis Report", key_col='norad_id')

output_path = '../data/clean/kinetic_master.csv'
master.to_csv(output_path, index=False)

# Read-back verification
export_check = pd.read_csv(output_path, dtype={'norad_id': str}, low_memory=False)
print(f"\n✅ EXPORT SUCCESS: {len(export_check):,} records saved to {output_path}")

# Kinetic Master Post-Synthesis Report

**Dimensions:** 68,282 rows × 48 columns

**Memory Footprint:** 84.11 MB

**Primary Key Check**: ✅ No duplicate norad_id values detected


---
### 📊 Data Quality Audit
| Column | Type | Nulls | Fill % | Status |
| :--- | :--- | :--- | :--- | :--- |
| **norad_id** | `object` | 0 | 100.0% | ✅ |
| **cospar_id** | `object` | 0 | 100.0% | ✅ |
| **object_name** | `object` | 0 | 100.0% | ✅ |
| **satellite_name** | `object` | 60,740 | 11.0% | ⚠️ |
| **official_name** | `object` | 60,740 | 11.0% | ⚠️ |
| **category** | `object` | 0 | 100.0% | ✅ |
| **object_type** | `object` | 0 | 100.0% | ✅ |
| **launch_mass_kg** | `float64` | 60,740 | 11.0% | ⚠️ |
| **proxy_mass_kg** | `float64` | 0 | 100.0% | ✅ |
| **dry_mass_kg** | `float64` | 0 | 100.0% | ✅ |
| **power_watts** | `float64` | 60,740 | 11.0% | ⚠️ |
| **rcs** | `float64` | 0 | 100.0% | ✅ |
| **rcs_class** | `object` | 0 | 100.0% | ✅ |
| **velocity_kms** | `float64` | 0 | 100.0% | ✅ |
| **kinetic_joules** | `float64` | 0 | 100.0% | ✅ |
| **semi_major_axis_km** | `float64` | 0 | 100.0% | ✅ |
| **proxy_power_watts** | `float64` | 0 | 100.0% | ✅ |
| **orbit_class** | `object` | 0 | 100.0% | ✅ |
| **orbit_type** | `object` | 60,740 | 11.0% | ⚠️ |
| **period_minutes** | `float64` | 0 | 100.0% | ✅ |
| **perigee_km** | `float64` | 0 | 100.0% | ✅ |
| **apogee_km** | `float64` | 0 | 100.0% | ✅ |
| **inclination_degrees** | `float64` | 0 | 100.0% | ✅ |
| **eccentricity** | `float64` | 0 | 100.0% | ✅ |
| **primary_purpose** | `object` | 60,740 | 11.0% | ⚠️ |
| **detailed_purpose** | `object` | 60,740 | 11.0% | ⚠️ |
| **users** | `object` | 60,740 | 11.0% | ⚠️ |
| **country_operator** | `object` | 60,740 | 11.0% | ⚠️ |
| **is_commercial** | `float64` | 60,740 | 11.0% | ⚠️ |
| **is_government** | `float64` | 60,740 | 11.0% | ⚠️ |
| **is_military** | `float64` | 60,740 | 11.0% | ⚠️ |
| **is_civil** | `float64` | 60,740 | 11.0% | ⚠️ |
| **geo_longitude** | `float64` | 0 | 100.0% | ✅ |
| **un_registry** | `object` | 60,740 | 11.0% | ⚠️ |
| **launch_date** | `object` | 0 | 100.0% | ✅ |
| **launch_year** | `int64` | 0 | 100.0% | ✅ |
| **sat_age_years** | `float64` | 0 | 100.0% | ✅ |
| **lifetime_years** | `float64` | 60,740 | 11.0% | ⚠️ |
| **launch_site** | `object` | 0 | 100.0% | ✅ |
| **ops_status** | `object` | 0 | 100.0% | ✅ |
| **data_status** | `object` | 67,024 | 1.8% | ⚠️ |
| **decay_date** | `object` | 33,347 | 51.2% | ⚠️ |
| **in_orbit** | `int64` | 0 | 100.0% | ✅ |
| **is_zombie** | `int64` | 0 | 100.0% | ✅ |
| **owner** | `object` | 0 | 100.0% | ✅ |
| **owner_code** | `object` | 0 | 100.0% | ✅ |
| **contractor** | `object` | 60,740 | 11.0% | ⚠️ |
| **contractor_country** | `object` | 60,740 | 11.0% | ⚠️ |

### 📝 Object Overview
|                    |   count |   unique | top                                                                                           |   freq |
|:-------------------|--------:|---------:|:----------------------------------------------------------------------------------------------|-------:|
| norad_id           |   68282 |    68282 | 1                                                                                             |      1 |
| cospar_id          |   68282 |    68282 | 1957-001A                                                                                     |      1 |
| object_name        |   68282 |    27200 | FENGYUN 1C DEB                                                                                |   3531 |
| satellite_name     |    7542 |     7527 | SB-WASS 3-5 (Space Based Wide Area Surveillance System) NOSS 3-5, USA 229, NRO L34, Intruder) |      2 |
| official_name      |    7542 |     7515 | Jilin-1                                                                                       |      5 |
| category           |   68282 |        5 | Debris                                                                                        |  35750 |
| object_type        |   68282 |        4 | DEBRIS                                                                                        |  35750 |
| rcs_class          |   68282 |        4 | UNKNOWN                                                                                       |  35351 |
| orbit_class        |   68282 |        5 | LEO                                                                                           |  61407 |
| orbit_type         |    7542 |        5 | Inclined                                                                                      |   4033 |
| primary_purpose    |    7542 |        7 | Communications                                                                                |   5514 |
| detailed_purpose   |    7542 |       54 | Not Specified                                                                                 |   6294 |
| users              |    7542 |       17 | Commercial                                                                                    |   6063 |
| country_operator   |    7542 |      104 | USA                                                                                           |   5154 |
| un_registry        |    7542 |       67 | USA                                                                                           |   4981 |
| launch_date        |   68282 |     5892 | 1999-05-10                                                                                    |   3537 |
| launch_site        |   68282 |       76 | PLMSC                                                                                         |  13481 |
| ops_status         |   68282 |        8 | DECAYED                                                                                       |  34935 |
| data_status        |    1258 |        2 | NEA                                                                                           |   1007 |
| decay_date         |   34935 |    14952 | 1976-08-02                                                                                    |    150 |
| owner              |   68282 |      129 | US                                                                                            |  27430 |
| owner_code         |   68282 |      129 | US                                                                                            |  27430 |
| contractor         |    7542 |      568 | SpaceX                                                                                        |   3987 |
| contractor_country |    7542 |       99 | USA                                                                                           |   5668 |

### 📈 Numeric Overview
|                     |   count |           mean |             std |            min |            25% |            50% |            75% |              max |
|:--------------------|--------:|---------------:|----------------:|---------------:|---------------:|---------------:|---------------:|-----------------:|
| launch_mass_kg      |    7542 |  690.292       |  5358.47        |    0.5         |  148           |  260           |  280           | 450000           |
| proxy_mass_kg       |   68282 |  396.227       |  1871.07        |    0.5         |   50           |   50           |  355           | 450000           |
| dry_mass_kg         |   68282 |  360.635       |  1727.2         |    0.472973    |   50           |   50           |  319.5         | 420000           |
| power_watts         |    7542 |  980.728       |  2786.24        |    0           |  120           |  120           |  815           |  84000           |
| rcs                 |   68282 |    2.27108     |    13.4992      |    0.0001      |    0.01        |    0.2815      |    1           |    928.31        |
| velocity_kms        |   68282 |    7.31058     |     1.0667      |    0.758409    |    7.4675      |    7.62624     |    7.75268     |     16.6766      |
| kinetic_joules      |   68282 |    9.62468e+09 |     5.33739e+10 |    1.44794e+07 |    1.45162e+09 |    1.51892e+09 |    1.03124e+10 |      1.32087e+13 |
| semi_major_axis_km  |   68282 | 8880.31        |  9290.42        | 1433.25        | 6631.84        | 6853.58        | 7148.05        | 692996           |
| proxy_power_watts   |   68282 |  159.163       |   975.632       |    0           |    0           |    0           |  163.846       |  84000           |
| period_minutes      |   68282 |  172.192       |   660.879       |    9           |   89.58        |   94.11        |  100.24        |  95687.7         |
| perigee_km          |   68282 | 1678.22        |  6447.85        |    5           |  207           |  442           |  629           | 314973           |
| apogee_km           |   68282 | 3319.27        | 13422.1         |   46           |  283           |  484           |  834           | 641287           |
| inclination_degrees |   68282 |   68.6296      |    24.7932      |    0           |   53           |   70           |   90.25        |    150.94        |
| eccentricity        |   68282 |    0.0876077   |     5.1792      |   -0.725044    |    0.000610302 |    0.00190967  |    0.00678512  |    575           |
| is_commercial       |    7542 |    0.828825    |     0.376687    |    0           |    1           |    1           |    1           |      1           |
| is_government       |    7542 |    0.101034    |     0.301394    |    0           |    0           |    0           |    0           |      1           |
| is_military         |    7542 |    0.0812782   |     0.27328     |    0           |    0           |    0           |    0           |      1           |
| is_civil            |    7542 |    0.0290374   |     0.167922    |    0           |    0           |    0           |    0           |      1           |
| geo_longitude       |   68282 |    0.189558    |     8.76415     | -179.8         |    0           |    0           |    0           |    359           |
| launch_year         |   68282 | 1998.55        |    20.0964      | 1957           | 1982           | 1998           | 2021           |   2026           |
| sat_age_years       |   68282 |   27.4416      |    20.0979      |    0           |    5           |   28           |   44           |     69           |
| lifetime_years      |    7542 |    5.14502     |     3.18728     |    0.25        |    4           |    4           |    4           |     30           |
| in_orbit            |   68282 |    0.488372    |     0.499868    |    0           |    0           |    0           |    1           |      1           |
| is_zombie           |   68282 |    0.182112    |     0.38594     |    0           |    0           |    0           |    0           |      1           |


✅ EXPORT SUCCESS: 68,282 records saved to ../data/clean/kinetic_master.csv
